# 삼각무역 데이터셋 구축
20260527

규제 이벤트(`regulation_events(2015~).csv`)별로 세 가지 무역 경로를 수집하여
우회수출 탐지용 데이터셋을 구성한다.

```
leg=0: 규제국 → 한국          (관세청)
leg=1: 규제국 → 후보국         (UN Comtrade)
leg=2: 후보국 → 한국           (관세청)
```

수집 순서: leg0 + leg2 (관세청) → 후보국 식별 → leg1 (UN Comtrade)

In [1]:
# 패키지 설치 (최초 1회)
%pip install comtradeapicall python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import time
import os
import requests
import xml.etree.ElementTree as ET
from datetime import datetime
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import comtradeapicall

pd.set_option('display.max_columns', None)

load_dotenv('반덤핑탐지 현석수정.env')
UN_COMTRADE_API_KEY = os.getenv('UN_COMTRADE_API_KEY')
GW_API_KEY          = os.getenv('GW_API_KEY')
BASE_DIR            = os.getenv('BASE_DIR', '.')

EVENTS_PATH = r'c:\Users\LG\Documents\카카오톡 받은 파일\regulation_events(2015~).csv'

In [3]:
# ── 관세청 수입 데이터 조회 ────────────────────────────────

def fetch_korea_imports(token, hs, start_ym, end_ym, cnty=None):
    """
    한국 기준 월별 수입 실적 조회 (관세청 공공데이터포털)
    cnty=None 이면 전체 국가 반환
    """
    _ENDPOINT = 'https://apis.data.go.kr/1220000/Itemtrade/getItemtradeList'
    _HS     = str(hs)
    _HS_LEN = len(_HS)
    if _HS_LEN not in {2, 4, 6, 10}:
        raise ValueError('hs는 2/4/6/10단위 코드여야 합니다.')

    # pd.period_range로 월 목록 생성 후 12개월씩 배치 분할
    months  = pd.period_range(start=start_ym, end=end_ym, freq='M').strftime('%Y%m').tolist()
    batches = [months[i:i + 12] for i in range(0, len(months), 12)]

    records = []
    for batch in batches:
        payload = {
            'serviceKey': token,
            'strtYymm':   batch[0],
            'endYymm':    batch[-1],
            'hsSgn':      _HS,
            'cntyCd':     cnty,
        }
        resp = requests.get(_ENDPOINT, params=payload, timeout=30)
        if resp.status_code != 200:
            raise RuntimeError(f'HTTP {resp.status_code}: {resp.text[:200]}')
        try:
            tree = ET.fromstring(resp.content)
        except ET.ParseError as exc:
            raise RuntimeError(f'XML 파싱 실패: {resp.text[:200]}') from exc
        if tree.findtext('./header/resultCode') != '00':
            raise RuntimeError(f"API 오류: {tree.findtext('./header/resultMsg')}")

        for node in tree.findall('./body/items/item'):
            if node.findtext('year') == '총계':
                continue
            records.append({
                'year':          node.findtext('year'),
                'cnty_name':     node.findtext('statCdCntnKor1'),
                'cnty_code':     node.findtext('statCd'),
                'hs_code':       node.findtext('hsCd'),
                'imp_value_usd': node.findtext('impDlr'),
                'imp_weight_kg': node.findtext('impWgt'),
            })

    _EMPTY = pd.DataFrame(columns=['year', 'cnty_name', 'cnty_code', 'hs_code', 'imp_value_usd', 'imp_weight_kg'])
    if not records:
        return _EMPTY

    df = pd.DataFrame(records)
    df['imp_value_usd'] = pd.to_numeric(df['imp_value_usd'], errors='coerce').fillna(0).astype(int)
    df['imp_weight_kg'] = pd.to_numeric(df['imp_weight_kg'], errors='coerce').fillna(0).astype(int)
    df['hs_code']       = df['hs_code'].astype(str).str[:_HS_LEN]

    df = (
        df.groupby(['year', 'cnty_name', 'cnty_code', 'hs_code'], as_index=False)
          .agg({'imp_value_usd': 'sum', 'imp_weight_kg': 'sum'})
    )
    return df[(df['imp_value_usd'] > 0) | (df['imp_weight_kg'] > 0)].reset_index(drop=True)

In [4]:
# ── UN Comtrade 수출 데이터 조회 ──────────────────────────

def fetch_comtrade_exports(token, reporter, partner, hs, period_range, freq='M', verbose=True):
    """
    UN Comtrade 수출 데이터 조회
    period_range: 'YYYYMM~YYYYMM' 형식
    """
    p_start, p_end = [x.strip() for x in period_range.split('~')]

    if freq == 'M':
        periods = pd.period_range(p_start, p_end, freq='M').strftime('%Y%m').tolist()
    else:
        periods = pd.period_range(p_start[:4], p_end[:4], freq='Y').strftime('%Y').tolist()

    batches = [periods[i:i + 12] for i in range(0, len(periods), 12)]
    frames  = []

    for idx, batch in enumerate(batches, 1):
        joined = ','.join(batch)
        if verbose:
            print(f'  [{idx}/{len(batches)}] UN Comtrade 조회 → {joined}')

        partial = comtradeapicall.getFinalData(
            token,
            typeCode='C',
            freqCode=freq,
            clCode='HS',
            period=joined,
            reporterCode=str(reporter),
            cmdCode=str(hs),
            flowCode='X',
            partnerCode=str(partner) if partner is not None else None,
            partner2Code=None,
            customsCode=None,
            motCode=None,
            maxRecords=2500,
            format_output='JSON',
            aggregateBy=None,
            breakdownMode='classic',
            countOnly=None,
            includeDesc=True,
        )

        if partial is None or not isinstance(partial, pd.DataFrame) or partial.empty:
            continue
        partial = partial.dropna(how='all').dropna(axis=1, how='all')
        if not partial.empty:
            frames.append(partial)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).drop_duplicates().reset_index(drop=True)

In [5]:
# ── 국가코드 매핑 테이블 ────────────────────────────────────
# regulation_events ISO3 → 관세청 GW 2자리 코드
ISO3_TO_GW = {
    'CHN': 'CN', 'JPN': 'JP', 'USA': 'US', 'KOR': 'KR',
    'IND': 'IN', 'VNM': 'VN', 'IDN': 'ID', 'MYS': 'MY',
    'THA': 'TH', 'TWN': 'TW', 'RUS': 'RU', 'BRA': 'BR',
    'CAN': 'CA', 'AUS': 'AU', 'GBR': 'GB', 'FRA': 'FR',
    'DEU': 'DE', 'ITA': 'IT', 'ESP': 'ES', 'MEX': 'MX',
    'TUR': 'TR', 'UKR': 'UA', 'ZAF': 'ZA', 'EGY': 'EG',
    'SAU': 'SA', 'ARE': 'AE', 'SGP': 'SG', 'HKG': 'HK',
    'FIN': 'FI', 'POL': 'PL', 'NLD': 'NL', 'BEL': 'BE',
    'SWE': 'SE', 'CHE': 'CH', 'NOR': 'NO', 'DNK': 'DK',
    'PRT': 'PT', 'GRC': 'GR', 'ROU': 'RO', 'CZE': 'CZ',
    'HUN': 'HU', 'PHL': 'PH', 'PAK': 'PK', 'BGD': 'BD',
    'NZL': 'NZ', 'CHL': 'CL', 'COL': 'CO', 'ARG': 'AR',
    'PER': 'PE', 'NGA': 'NG', 'KEN': 'KE', 'MAR': 'MA',
    'IRN': 'IR', 'IRQ': 'IQ', 'ISR': 'IL', 'KWT': 'KW',
    'QAT': 'QA', 'JOR': 'JO', 'KAZ': 'KZ', 'UZB': 'UZ',
    'BLR': 'BY', 'AZE': 'AZ', 'GEO': 'GE', 'LTU': 'LT',
    'LVA': 'LV', 'EST': 'EE', 'SRB': 'RS', 'HRV': 'HR',
    'SVN': 'SI', 'SVK': 'SK', 'BGR': 'BG',
}

# 관세청 GW 2자리 코드 → UN Comtrade numeric 코드
GW_TO_UN = {
    'CN': 156, 'JP': 392, 'US': 842, 'KR': 410,
    'IN': 356, 'VN': 704, 'ID': 360, 'MY': 458,
    'TH': 764, 'TW': 490, 'RU': 643, 'BR':  76,
    'CA': 124, 'AU':  36, 'GB': 826, 'FR': 250,
    'DE': 276, 'IT': 381, 'ES': 724, 'MX': 484,
    'TR': 792, 'UA': 804, 'ZA': 710, 'EG': 818,
    'SA': 682, 'AE': 784, 'SG': 702, 'HK': 344,
    'FI': 246, 'PL': 616, 'NL': 528, 'BE':  56,
    'SE': 752, 'CH': 756, 'NO': 578, 'DK': 208,
    'PT': 620, 'GR': 300, 'RO': 642, 'CZ': 203,
    'HU': 348, 'PH': 608, 'PK': 586, 'BD':  50,
    'NZ': 554, 'CL': 152, 'CO': 170, 'AR':  32,
    'PE': 604, 'NG': 566, 'KE': 404, 'MA': 504,
    'IR': 364, 'IQ': 368, 'IL': 376, 'KW': 414,
    'QA': 634, 'JO': 400, 'KZ': 398,
}

# 관세청 GW 2자리 코드 → 한글 국가명
GW_TO_KR = {
    'CN': '중국', 'JP': '일본', 'US': '미국', 'KR': '한국',
    'IN': '인도', 'VN': '베트남', 'ID': '인도네시아', 'MY': '말레이시아',
    'TH': '태국', 'TW': '대만', 'RU': '러시아', 'BR': '브라질',
    'CA': '캐나다', 'AU': '호주', 'GB': '영국', 'FR': '프랑스',
    'DE': '독일', 'IT': '이탈리아', 'ES': '스페인', 'MX': '멕시코',
    'TR': '튀르키예', 'UA': '우크라이나', 'ZA': '남아프리카', 'EG': '이집트',
    'SA': '사우디아라비아', 'AE': '아랍에미리트', 'SG': '싱가포르', 'HK': '홍콩',
    'FI': '핀란드', 'PL': '폴란드', 'NL': '네덜란드', 'BE': '벨기에',
    'SE': '스웨덴', 'CH': '스위스', 'NO': '노르웨이', 'DK': '덴마크',
    'PT': '포르투갈', 'GR': '그리스', 'RO': '루마니아', 'CZ': '체코',
    'HU': '헝가리', 'PH': '필리핀', 'PK': '파키스탄', 'BD': '방글라데시',
    'NZ': '뉴질랜드', 'CL': '칠레', 'CO': '콜롬비아', 'AR': '아르헨티나',
    'PE': '페루', 'NG': '나이지리아', 'KE': '케냐', 'MA': '모로코',
    'IR': '이란', 'IQ': '이라크', 'IL': '이스라엘', 'KW': '쿠웨이트',
    'QA': '카타르', 'JO': '요르단', 'KZ': '카자흐스탄',
}

In [6]:
# ── 규제 이벤트 로드 및 HS 코드 분리 ────────────────────────

def load_events(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    rows = []
    for _, row in df.iterrows():
        hs_list = [h.strip() for h in str(row['hs_code']).split(';') if h.strip()]
        for hs in hs_list:
            r = row.copy()
            r['hs_code'] = hs
            rows.append(r)
    return pd.DataFrame(rows).reset_index(drop=True)


events_df = load_events(EVENTS_PATH)
print(f'총 {len(events_df)}행  |  source_row_id: {events_df["source_row_id"].nunique()}개  |  event_id: {events_df["event_id"].nunique()}개')
events_df[['event_id','source_row_id','origin_country_iso3','hs_code','product_name_normalized','start_date','end_date']].head(8)

총 155행  |  source_row_id: 62개  |  event_id: 104개


,event_id,source_row_id,origin_country_iso3,hs_code,product_name_normalized,start_date,end_date
0,AD-01-01,1,CHN,700529,플로트판유리,2015-01-07,2018-01-06
1,AD-02-01,2,CHN,690721,도자기질 타일,2015-02-25,2018-02-24
2,AD-02-01,2,CHN,690722,도자기질 타일,2015-02-25,2018-02-24
3,AD-02-01,2,CHN,690723,도자기질 타일,2015-02-25,2018-02-24
4,AD-03-01,3,CHN,721633,H형강,2015-07-30,2020-07-29
5,AD-04-01,4,JPN,848120,공기압 전송용 밸브,2015-08-19,2020-08-18
6,AD-05-01,5,CHN,291531,초산에틸,2015-11-19,2018-11-18
7,AD-05-02,5,SGP,291531,초산에틸,2015-11-19,2018-11-18


In [7]:
# ── 유틸 함수 ────────────────────────────────────────────

def analysis_window(start_date_str):
    """start_date 기준 전후 12개월 → (start_ym, end_ym) YYYYMM 반환"""
    dt = pd.to_datetime(start_date_str)
    return (
        (dt - pd.DateOffset(months=12)).strftime('%Y%m'),
        (dt + pd.DateOffset(months=12)).strftime('%Y%m'),
    )


def _parse_ym(year_series):
    """관세청 year 컬럼(YYYY.MM 또는 YYYYMM) → YYYY-MM 문자열"""
    s = year_series.astype(str)
    if s.str.contains('.', regex=False).any():
        return s.str.replace('.', '-', regex=False)
    return s.str[:4] + '-' + s.str[4:6]


# 최종 데이터셋 컬럼 정의
SCHEMA = [
    'source_row_id', 'event_id', 'intermediary_num', 'leg',
    'regulated_cnty', 'intermediary_cnty', 'destination_cnty',
    'trade_date', 'hs_code', 'product_name',
    'quantity', 'value_dlr', 'unit_price',
    'regulation_start', 'regulation_end',
]

In [8]:
# ── 원본 데이터 → 최종 스키마 변환 ────────────────────────────

def _gw_as_flow(gw_df, leg, regulated_gw, via_gw,
                hs_code, product_name, reg_start, reg_end):
    """fetch_korea_imports 결과 → SCHEMA 구조 DataFrame"""
    if gw_df is None or gw_df.empty:
        return pd.DataFrame(columns=SCHEMA)

    out = pd.DataFrame()
    out['trade_date']        = _parse_ym(gw_df['year'])
    out['hs_code']           = hs_code
    out['product_name']      = product_name
    kg  = pd.to_numeric(gw_df['imp_weight_kg'], errors='coerce')
    usd = pd.to_numeric(gw_df['imp_value_usd'], errors='coerce')
    out['quantity']          = kg
    out['value_dlr']         = usd
    out['unit_price']        = (usd / kg.replace(0, np.nan)).round(6)
    out['leg']               = leg
    out['regulated_cnty']    = regulated_gw
    out['intermediary_cnty'] = via_gw
    out['destination_cnty']  = 'KR'
    out['regulation_start']  = reg_start
    out['regulation_end']    = reg_end
    return out.reset_index(drop=True)


def _ct_as_flow(ct_df, regulated_gw, via_gw,
                hs_code, product_name, reg_start, reg_end):
    """fetch_comtrade_exports 결과 → SCHEMA 구조 DataFrame"""
    if ct_df is None or ct_df.empty:
        return pd.DataFrame(columns=SCHEMA)

    out = pd.DataFrame()
    period_str = ct_df['period'].astype(str).str.replace('.0', '', regex=False).str.strip()
    out['trade_date']        = pd.to_datetime(period_str, format='%Y%m').dt.strftime('%Y-%m')
    out['hs_code']           = hs_code
    out['product_name']      = product_name
    if 'netWgt' in ct_df.columns:
        kg = pd.to_numeric(ct_df['netWgt'], errors='coerce')
    elif 'qty' in ct_df.columns:
        kg = pd.to_numeric(ct_df['qty'], errors='coerce')
    else:
        kg = pd.Series(np.nan, index=ct_df.index)
    usd = pd.to_numeric(ct_df.get('primaryValue', pd.Series(np.nan, index=ct_df.index)), errors='coerce')
    out['quantity']          = kg
    out['value_dlr']         = usd
    out['unit_price']        = (usd / kg.replace(0, np.nan)).round(6)
    out['leg']               = 1
    out['regulated_cnty']    = regulated_gw
    out['intermediary_cnty'] = via_gw
    out['destination_cnty']  = 'KR'
    out['regulation_start']  = reg_start
    out['regulation_end']    = reg_end
    return out.reset_index(drop=True)

In [9]:
# ── 단일 이벤트 데이터셋 구축 ──────────────────────────────

def collect_event(event_row, verbose=True):
    source_row_id  = event_row['source_row_id']
    event_id       = event_row['event_id']
    regulated_iso3 = event_row['origin_country_iso3']
    hs_code        = str(event_row['hs_code']).strip()
    product_name   = event_row['product_name_normalized']
    reg_start      = str(event_row['start_date'])
    reg_end        = str(event_row['end_date'])

    regulated_gw = ISO3_TO_GW.get(regulated_iso3)
    if regulated_gw is None:
        print(f'[스킵] GW 코드 없음: {regulated_iso3} ({event_id})')
        return pd.DataFrame(columns=SCHEMA)

    regulated_un = GW_TO_UN.get(regulated_gw)

    win_start, win_end = analysis_window(reg_start)
    if verbose:
        print(f'[{event_id}] {product_name} | 규제국={regulated_iso3}({regulated_gw}) | HS={hs_code} | 기간={win_start}~{win_end}')

    korea_imp = fetch_korea_imports(
        token=GW_API_KEY,
        hs=hs_code,
        start_ym=win_start,
        end_ym=win_end,
    )

    if korea_imp.empty:
        if verbose:
            print('  관세청 수입 데이터 없음')
        return pd.DataFrame(columns=SCHEMA)

    korea_imp = korea_imp[
        korea_imp['cnty_code'].notna() &
        ~korea_imp['cnty_code'].isin({'KR', '한국', ''})
    ].copy()

    regulated_rows = korea_imp[korea_imp['cnty_code'] == regulated_gw].copy()
    candidate_rows = {
        code: grp.copy()
        for code, grp in korea_imp[korea_imp['cnty_code'] != regulated_gw].groupby('cnty_code')
    }
    candidates = sorted(candidate_rows.keys())

    if verbose:
        print(f'  leg0: {len(regulated_rows)}행 | 후보국: {len(candidates)}개')

    frames = []

    for num, via_gw in enumerate(candidates, start=1):
        via_un = GW_TO_UN.get(via_gw)

        leg0 = _gw_as_flow(regulated_rows, 0, regulated_gw, via_gw,
                           hs_code, product_name, reg_start, reg_end)
        leg0['intermediary_num'] = num

        leg2 = _gw_as_flow(candidate_rows[via_gw], 2, regulated_gw, via_gw,
                           hs_code, product_name, reg_start, reg_end)
        leg2['intermediary_num'] = num

        leg1 = pd.DataFrame(columns=SCHEMA)
        if regulated_un is not None and via_un is not None:
            ct_raw = fetch_comtrade_exports(
                token=UN_COMTRADE_API_KEY,
                reporter=regulated_un,
                partner=via_un,
                hs=hs_code,
                period_range=f'{win_start}~{win_end}',
                freq='M',
                verbose=verbose,
            )
            leg1 = _ct_as_flow(ct_raw, regulated_gw, via_gw,
                               hs_code, product_name, reg_start, reg_end)
            leg1['intermediary_num'] = num
        else:
            if verbose:
                print(f'  [leg1 스킵] UN 코드 없음: {via_gw}')

        frames.extend([leg0, leg1, leg2])

    if not frames:
        return pd.DataFrame(columns=SCHEMA)

    df = pd.concat(frames, ignore_index=True)
    df['source_row_id'] = source_row_id
    df['event_id']      = event_id

    return (
        df[SCHEMA]
        .sort_values(['source_row_id', 'event_id', 'intermediary_num', 'trade_date', 'leg'])
        .reset_index(drop=True)
    )

## 테스트: AD-01-01 (플로트판유리, CHN, HS=700529)

In [10]:
test_row = events_df[events_df['event_id'] == 'AD-01-01'].iloc[0]
print(test_row[['event_id','source_row_id','origin_country_iso3','hs_code','product_name_normalized','start_date']].to_string())
print()

event_id                     AD-01-01
source_row_id                       1
origin_country_iso3               CHN
hs_code                        700529
product_name_normalized        플로트판유리
start_date                 2015-01-07



In [11]:
test_result = collect_event(test_row, verbose=True)
print(f'\n결과: {len(test_result)}행, 후보국 {test_result["intermediary_num"].nunique() if not test_result.empty else 0}개')
test_result

[AD-01-01] 플로트판유리 | 규제국=CHN(CN) | HS=700529 | 기간=201401~201601
  관세청 수입 데이터 없음

결과: 0행, 후보국 0개


,source_row_id,event_id,intermediary_num,leg,regulated_cnty,intermediary_cnty,destination_cnty,trade_date,hs_code,product_name,quantity,value_dlr,unit_price,regulation_start,regulation_end


In [12]:
if not test_result.empty:
    print('=== leg별 행수 ===')
    print(test_result.groupby('leg').size().to_string())
    print()
    print('=== 상위 후보국 (leg2 value_dlr 합계 기준) ===')
    top_via = (
        test_result[test_result['leg'] == 2]
        .groupby('intermediary_cnty')['value_dlr']
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )
    print(top_via.to_string())

## 전체 이벤트 수집

> **주의**: UN Comtrade API는 호출 횟수 제한이 있습니다.  
> 후보국 수에 따라 이벤트당 수십 회 호출이 발생할 수 있으므로 `max_events`로 조절하세요.
> 중간 결과는 자동 저장됩니다.

In [13]:
import json
from pathlib import Path

OUTPUT_PATH     = os.path.join(BASE_DIR, 'triangular_trade_dataset.csv')
CHECKPOINT_PATH = os.path.join(BASE_DIR, 'triangular_trade_checkpoint.json')


def collect_all(events_df, max_events=None, sleep_sec=2.0, verbose=True):
    done = set()
    if Path(CHECKPOINT_PATH).exists():
        with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
            done = set(json.load(f))
        print(f'체크포인트 로드: {len(done)}개 완료됨')

    accumulated = []
    if Path(OUTPUT_PATH).exists():
        accumulated.append(pd.read_csv(OUTPUT_PATH, encoding='utf-8-sig'))
        print(f'기존 결과 로드: {len(accumulated[0])}행')

    n = 0
    for _, row in events_df.iterrows():
        key = f"{row['event_id']}|{row['hs_code']}"
        if key in done:
            continue
        if max_events is not None and n >= max_events:
            print(f'max_events={max_events} 도달, 중단')
            break

        try:
            chunk = collect_event(row, verbose=verbose)
            if not chunk.empty:
                accumulated.append(chunk)
        except Exception as exc:
            print(f'[오류] {key}: {exc}')

        done.add(key)
        n += 1

        with open(CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
            json.dump(list(done), f, ensure_ascii=False)

        if accumulated:
            merged = pd.concat(accumulated, ignore_index=True).drop_duplicates()
            merged.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

        if sleep_sec > 0:
            time.sleep(sleep_sec)

    result = (
        pd.concat(accumulated, ignore_index=True).drop_duplicates().reset_index(drop=True)
        if accumulated else pd.DataFrame(columns=SCHEMA)
    )
    result = result.sort_values(
        ['source_row_id', 'event_id', 'intermediary_num', 'trade_date', 'leg']
    ).reset_index(drop=True)
    result.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f'\n완료: {len(result)}행 → {OUTPUT_PATH}')
    return result

In [ ]:
# 처음 3개 이벤트로 테스트 실행
# result = collect_all(events_df, max_events=3, sleep_sec=1.0)

In [ ]:
# 전체 실행 (API 할당량 확인 후 주석 해제)
# result = collect_all(events_df, max_events=None, sleep_sec=2.0)